In [1]:
# importing the relevant libraries
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.dummy import DummyClassifier

In [8]:
# importing the dataset as csv
# I cleaned the data with sql so i exported it and saved in my file explorer so i have imported it
df = pd.read_csv(r"C:\Users\Administrator\sales_202610292013.csv")

df.head()

,customer_id,transaction_date,product_name,product_category,store_id,quantity,unit_price,total_amount,payment_method,promotion_active,is_holiday,inflation_index
0,C00085484,2025-04-07,Product_1459,Hair Care,S00000091,8,3.66,29.28,M-Pesa,No,Yes,2.62
1,C00027234,2024-03-05,Product_3904,Skin Care,S00000154,6,16.15,96.90,Credit Card,No,No,2.17
2,C00062273,2026-03-02,Product_462,Meat,S00000190,5,10.52,52.60,M-Pesa,No,No,2.27
3,C00021532,2024-09-02,Product_650,Baby Food,S00000029,3,23.13,69.39,M-Pesa,Yes,No,1.48
4,C00020196,2025-03-04,Product_1158,Bedding,S00000134,3,3.81,11.43,M-Pesa,No,No,2.43


Grouping the data

In [10]:
# Group the data by customer to get their summary stats
df['transaction_date']= pd.to_datetime(df['transaction_date'])

rfm = df.groupby('customer_id').agg({
    'transaction_date':'max',
    'total_amount': ['count', 'sum']
})

rfm.columns = ['last_purchase','frequency','monetary']
rfm = rfm.reset_index()
rfm.head()

,customer_id,last_purchase,frequency,monetary
0,C00000001,2025-07-01,2,264.22
1,C00000002,2023-07-11,1,350.20
2,C00000003,2024-11-02,1,60.03
3,C00000004,2023-10-05,1,103.68
4,C00000005,2024-04-09,1,48.66


In [11]:
# Created a column called 'churned'.So If they didn't buy since Jan 2024,they churned =1 .
rfm['churned'] = (rfm['last_purchase'] < '2024-01-01').astype(int)
rfm['churned'].value_counts()

churned
0    66226
1     9573
Name: count, dtype: int64

The model needs a target to predict.We are defining 'churn' as any customer who hasn't made a purchase since January 1st 2024.If their last purchase was before that date they get a 1 (churned).If they bought after they get a 0 (active)

In [12]:
# Separate the features (X) from the target (y)
X = rfm[['frequency','monetary']]
y = rfm['churned']

In [14]:
# Split the data:80% for training and 20% for testing
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

We put the data we want the model to learn from frequency and monetary value into X.The answer key whether churned or not goes into y.We then split it so the model learns on 80% of the data and is tested on the remaining 20% to see if it actually works.

In [15]:
# Baseline model:it just guesses the most common answer
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train,y_train)
base_pred = baseline.predict(X_test)

print(f"Baseline Accuracy: {accuracy_score(y_test, base_pred):.2f}")

Baseline Accuracy: 0.87


Before building a smart model we build a "dumb" baseline model.This gives us a number to beat.

In [18]:
# Random Forest model:where it learns patterns from the data
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train,y_train)
rf_pred = rf_model.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred):.2f}")

Random Forest Accuracy: 0.84


This is the real model.A Random Forest builds multiple decision trees to find patterns.It then tests its predictions on the 20% of data it hasn't seen before.

In [19]:
# Show detailed metrics
print(classification_report(y_test,rf_pred))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91     13251
           1       0.18      0.08      0.12      1909

    accuracy                           0.84     15160
   macro avg       0.53      0.51      0.51     15160
weighted avg       0.79      0.84      0.81     15160



I built a Customer Churn Predictive Model in to identify which shoppers are at risk of leaving the retail chain.After exporting the cleaned data from PostgreSQL,I grouped the transactions by customer to calculate their purchase frequency,total money value and last purchase date,defining "churn" as not buying since January 2024.I split the data 80/20 and trained a Random Forest Classifier comparing its accuracy against a simple baseline model that just guessed the most common outcome.While the Random Forest model beat the baseline,it sometimes struggles to predict churn accurately for new customers with only one purchase because their limited history makes it hard for the model to distinguish a one-time buyer from a loyal customer who just started shopping recently.